In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, IterableDataset
import sys
print(torch.cuda.is_available())
sys.path.append("/mnt/home/lserrano/disco-ball/")

import numpy as np
import random
import matplotlib.pyplot as plt
import h5py
import os
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR # A more standard scheduler
from einops import rearrange # A more standard import
from tqdm import tqdm
import torch.nn.functional as F

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_prediction_comparison(pred, ground_truth, idx=0, figsize=(18, 6), 
                             style='seaborn-v0_8', save_path=None, 
                             title_prefix=None, show_plot=True):
    """
    Create a three-panel plot: Predictions | Ground Truth | Delta (Difference)
    
    Parameters:
    -----------
    pred : torch.Tensor or numpy.ndarray
        Prediction tensor with shape [samples, time_steps, features] or [samples, features]
    ground_truth : torch.Tensor or numpy.ndarray
        Ground truth data with same structure as pred
    idx : int, default=0
        Index of the sample to plot
    figsize : tuple, default=(18, 6)
        Figure size (width, height)
    style : str, default='seaborn-v0_8'
        Matplotlib style to use
    save_path : str, optional
        Path to save the plot
    title_prefix : str, optional
        Prefix for the main title
    show_plot : bool, default=True
        Whether to display the plot
    
    Returns:
    --------
    fig, axes : matplotlib figure and axes objects
    """
    
    # Set style
    plt.style.use(style)
    
    # Create figure with 3 subplots side by side
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Convert tensors to numpy if needed
    def to_numpy(tensor):
        if hasattr(tensor, 'cpu'):
            return tensor.cpu().detach().numpy()
        return tensor
    
    delta = pred - ground_truth
        
    for t in range(pred.shape[1]):
        pred_t = to_numpy(pred[idx, t].squeeze())
        gt_t = to_numpy(ground_truth[idx, t].squeeze())
        delta_t = to_numpy(delta[idx, t].squeeze())
        
        # Plot each time step
        axes[0].plot(pred_t, label=f't={t}', alpha=0.7, linewidth=2)
        axes[1].plot(gt_t, label=f't={t}', alpha=0.7, linewidth=2)
        axes[2].plot(delta_t, label=f't={t}', alpha=0.7, linewidth=2)
    
    # Customize each subplot
    titles = ['Predictions', 'Ground Truth', 'Delta (Pred - GT)']
    
    for i, (ax, title) in enumerate(zip(axes, titles)):
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_xlabel('Time Steps', fontsize=12)
        ax.set_ylabel('Values', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    # Add horizontal line at y=0 for delta plot
    axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
    
    # Set main title
    if title_prefix:
        fig.suptitle(f'{title_prefix} - Sample {idx}', fontsize=16, fontweight='bold')
    else:
        fig.suptitle(f'Prediction Analysis - Sample {idx}', fontsize=16, fontweight='bold')
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    
    # Save if path provided
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    # Show plot
    if show_plot:
        plt.show()
            
    return fig, axes

In [ ]:
from models import DISCOHouse
from src.advection_diffusion import Fractaloid
from train.train import DISCOLitModule, advection_diffusion_analytical
from src.plot_dataset_samples import plot_prediction_vs_ground_truth

In [ ]:
class RelativeL2(nn.Module):
    def forward(self, x, y, aggregate="mean"):
        x = rearrange(x, "b ... -> b (...)")
        y = rearrange(y, "b ... -> b (...)")
        diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
        y_norms = torch.linalg.norm(y, ord=2, dim=-1)

        if aggregate == "mean":
            return (diff_norms / y_norms).mean()
        else:
            return (diff_norms / y_norms)

In [ ]:
def autoregressive_predict(model, initial_seq, n_pred, device):
    preds = []
    current = initial_seq.clone().to(device)
    n_input = current.shape[1]
    for t in range(n_pred):
        inp = current[:, -n_input:].to(device)
        with torch.no_grad():
            state_labels = torch.tensor([0], device=inp.device)
            next_frame, metadata = model(inp, state_labels, n_future_steps=1)
            if t == 0:
                theta = metadata['theta_latent']
        current = torch.cat([current, next_frame], axis=1)
        preds.append(next_frame)
    return torch.cat(preds, axis=1), theta

In [ ]:
class TemporalBatchDatasetFly(IterableDataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=50.0,
                 v_range=(0.01, 1.0), D_range=(0.01, 1.0),
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        for _ in range(self.n_batches):
            input_frames = self.input_frames
            batch_inputs = []
            batch_targets = []
            batch_v = []
            batch_d = []
            batch_init = []
            for _ in range(self.batch_size):
                # Sample advection speed and viscosity
                if self.split == 'train':
                    if random.random() < 0.5:
                        v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                        D = 0
                    else:
                        v = 0
                        D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                else:
                    v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                    D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                # Generate fractaloid initial condition
                fractaloid = Fractaloid(
                    degree=self.fractal_degree,
                    power=self.fractal_power,
                    size=self.nx,
                    patch_size=self.nx
                )
                u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
                u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=D, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                max_start_index_input = u_xt.shape[0] - input_frames
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(D)
                batch_init.append(torch.from_numpy(u0))
            batch = {
                'input': torch.stack(batch_inputs),
                'target': torch.stack(batch_targets),
                'velocities': batch_v,
                'diffusivities': batch_d,
                'initial_conditions': torch.stack(batch_init)
            }
            yield batch

In [ ]:
batch_size=128
sub_x=1
sub_t=1
n_input_frames=16
n_output_frames=50-16 #50-n_input_frames
relative_l2_error = RelativeL2()

In [ ]:
n_batches = int(1000//batch_size)  # or set as needed for your epoch size
split="train"
train_ds = TemporalBatchDatasetFly(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    fractal_power=3.0,
    fractal_degree=256, # nx
    v_range=(0.01, 1.0),
    D_range=(0.001, 1.0),
)
train_loader = DataLoader(train_ds, batch_size=None, num_workers=4, prefetch_factor=4, pin_memory=True)

In [ ]:
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
device="cuda" if torch.cuda.is_available() else "cpu"
#ckpt_time="DISCO_advection-diffusion_adjFalse_h128_t2_steps10_bs64_lr0.0005_ctxFalse_inframes16_outframes2_T10"
ckpt_time = "DISCO_advection-diffusion_solverrk4_adjFalse_h128_t2_steps1_initFalse_bs64_lr0.0005_ctxTrue_noise0.0001_inframes16_outframes2_T10"
ckpt_path = f"/mnt/home/lserrano/disco-ball/outputs/{ckpt_time}/best-checkpoint.ckpt" #last.ckpt" otherwise
#ckpt_time = 


#ckpt_path = f"/mnt/home/lserrano/disco-ball/outputs/{ckpt_time}/last-v2.ckpt" #last.ckpt" otherwise
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
print(f"Loading model from {ckpt_path}...")
model = DISCOLitModule.load_from_checkpoint(ckpt_path, map_location=device)
model = model.model.to(device)
model.eval()

# I. Finetune theta

In [ ]:
results_dir = f"results/{ckpt_time}"
dataset_name="advection_diffusion"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/plots/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/predictions/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/theta/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/errors/", exist_ok=True)
#output_path = f"plots/{ckpt_time}"
#output_path = f"plots/{setting}/baseline/"

In [ ]:
n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(train_loader):
    inp, target = batch["input"], batch["target"]
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    break

In [ ]:
    
inp = inp.to(device)
target = target.to(device)
state_labels = torch.tensor([0], device=inp.device)

x_shape = inp.shape
B, T, C = x_shape[:3]
spatial = x_shape[3:]
dim = len(spatial)

n_sample = inp.shape[0]
#pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

# encode into 2 dimensional
theta_latent, metadata= model.encode_theta_latent(inp, state_labels)

# decode into 100k parameters
theta = model.decode_theta(theta_latent, dim)

#predict
x_test_ = inp[:, -1]
pred_test = []
with torch.no_grad():
    #pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=34, predict_normed=False, metadata=metadata)
    for _ in range(n_output_frames):
        pred, _ = model.solve_ode(x_test_, theta, state_labels, dim, integration_time=1, n_future_steps=1, predict_normed=False, metadata=metadata)
        pred_test.append(pred[:, -1])
        x_test_ = pred[:, -1]

pred_test = torch.stack(pred_test, 1)
rollout_error = relative_l2_error(pred_test, target).item()

print(f"Initial error", rollout_error)

In [ ]:
target.shape

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
epochs = 100

# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
x_test = inp[:, -1].to(device)
y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
theta = theta.clone().detach().requires_grad_()
optimizer = torch.optim.AdamW([theta], lr=1e-3)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-2)
    x_train = inp[:, t]
    y_train = inp[:, t+1]
    
    # Run the model to get the prediction.
    x_train_ = x_train.clone()
    pred_train, metadata = model.solve_ode(
    x_train_, theta, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)

    # Calculate the training loss.

    loss = relative_l2_error(pred_train, y_train)
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()

            pred_test = []
            # Run the model on the test data.
            x_test_ = x_test.clone()
            pred_test, _ = model.solve_ode(
                x_test_, theta, state_labels, dim, n_future_steps=n_output_frames, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
                )

            #pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
pred_test = pred_test.cpu().detach()
y_test = y_test.cpu().detach()

In [ ]:
pred_test.shape

In [ ]:
fig, ax = plot_prediction_comparison(pred_test.squeeze(1), y_test.squeeze(1), idx=9)

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
epochs = 1000

# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
x_test = inp[:, -1].to(device)
y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
theta1 = (0.1*torch.randn_like(theta)).clone().detach().requires_grad_()
theta2 = (0.1*torch.randn_like(theta)).clone().detach().requires_grad_()
optimizer = torch.optim.AdamW([theta1]+ [theta2], lr=1e-3)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-2)
    x_train = inp[:, t]
    y_train = inp[:, t+1]
    
    # Run the model to get the prediction.
    pred1, metadata = model.solve_ode(
        x_train, theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata
    )
    pred_train, metadata = model.solve_ode(
        pred1[:,-1], theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata
    )
    
    # Calculate the training loss.
    loss = relative_l2_error(pred_train, y_train)
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()

            pred_test = []
            # Run the model on the test data.
            x_test_ = x_test.clone()
            for _ in range(n_output_frames):
                pred1, _ = model.solve_ode(
                x_test_, theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
                )
                pred, _ = model.solve_ode(
                pred1[:, -1], theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
                )
                pred_test.append(pred)
                x_test_ = pred[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )
    
    

print("Training finished.")

In [ ]:
fig, ax = plot_prediction_comparison(pred_test.squeeze(1), y_test.squeeze(1), idx=9)

# II. Finetune latent theta

In [ ]:
n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(train_loader):
    inp, target = batch["input"], batch["target"]
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    break
  

In [ ]:
inp = inp.to(device)
target = target.to(device)
state_labels = torch.tensor([0], device=inp.device)

x_shape = inp.shape
B, T, C = x_shape[:3]
spatial = x_shape[3:]
dim = len(spatial)

n_sample = inp.shape[0]
#pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

# encode into 2 dimensional
theta_latent, metadata= model.encode_theta_latent(inp, state_labels)

# decode into 100k parameters
theta = model.decode_theta(theta_latent, dim)

#predict
pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=34, predict_normed=False, metadata=metadata)
rollout_error = relative_l2_error(pred, target).item()

print(f"Initial error", rollout_error)

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx=0
epochs = 1000

# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
x_test = inp[:, -1].to(device)
y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
theta_latent = theta_latent.clone().detach().requires_grad_()
optimizer = torch.optim.AdamW([theta_latent], lr=0.1)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()

            theta = model.decode_theta(theta_latent, dim)
            pred_test = []
            x_test_ = x_test.clone()
            for _ in range(n_output_frames):
                pred, _ = model.solve_ode(
                x_test_, theta, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
                )
                pred_test.append(pred)
                x_test_ = pred[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
    
            
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Theta latent{idx}: {theta_latent[idx], theta_latent[idx].grad}"
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )
    
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-2)
    x_train = inp[:, t]
    y_train = inp[:, t+1]

    theta = model.decode_theta(theta_latent, dim)
    
    # Run the model to get the prediction.
    pred_train, metadata = model.solve_ode(
        x_train, theta, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata
    )
    
    # Calculate the training loss.
    loss = relative_l2_error(pred_train, y_train)
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    #print('thetagrad', theta_latent.grad)
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    

print("Training finished.")

# II. manual composition: i.e. what we want to reach by optimization

In [ ]:
class TemporalDatasetFixedCI(torch.utils.data.Dataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=50.0,
                 v_range=[0.01, 0.025, 0.05, 0.1, 0.5, 1.0], D_range=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.u0 = []
        for _ in range(self.batch_size):
            fractaloid = Fractaloid(
                degree=self.fractal_degree,
                power=self.fractal_power,
                size=self.nx,
                patch_size=self.nx
            )
            u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
            u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
            self.u0.append(torch.from_numpy(u0))
            
        self.u0 = torch.stack(self.u0)

    def __len__(self):
        return len(self.u0)

    def __getitem__(self, idx):
    
        input_frames = self.input_frames
        batch_inputs = []
        batch_targets = []
        batch_v = []
        batch_d = []
        batch_init = []
        for v in self.v_range:
            for d in self.D_range:
                u0 = self.u0[idx]
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=d, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(d)
                batch_init.append(u0)
                
        batch = {
            'input': torch.stack(batch_inputs),
            'target': torch.stack(batch_targets),
            'velocities': batch_v,
            'diffusivities': batch_d,
            'initial_conditions': torch.stack(batch_init)
        }
        return batch

In [ ]:
n_batches = 1  # or set as needed for your epoch size
batch_size = 1
split="test"
n_input_frames=16
n_output_frames=34
model.eval()

#composition_type="composition"
#composition_type="sum"

#advection_speeds = [1.0, 0., 1.0]
#viscosities = [0., 0.9, 0.9]

#advection_speeds = [0.9, 0.9, 1.8]

#advection_speeds = [0.9, 0.9, 1.8]
#viscosities = [0., 0, 0.]

advection_speeds = [0., 0., 0.]
viscosities = [0.95, 0.95, 1.9]


all_velocities = []
all_diffusivities = []
all_input = []
all_target = []
all_theta_latent = []
all_theta = []

test_ds = TemporalDatasetFixedCI(
        n_batches=n_batches,
        batch_size=batch_size,
        sub_x=sub_x,
        sub_t=sub_t,
        split=split,
        input_frames=n_input_frames,
        output_frames=n_output_frames,
        L=16.0,
        nx=256,
        nt=100,
        T=10.0,
        fractal_power=3.0,
        fractal_degree=256, # nx
        v_range=[0],#(0.01, 1.0),
        D_range=[0], #(0.001, 1.0),
    )


for advection_speed, viscosity in zip(advection_speeds, viscosities):

    test_ds.v_range=[advection_speed]
    test_ds.D_range=[viscosity]
    test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=1, prefetch_factor=1, pin_memory=True, shuffle=False)
    
    for batch in tqdm(test_loader):
        inp, target = batch["input"], batch["target"]
        inp = inp.squeeze(1)
        target = target.squeeze(1)
        print('inp', inp.shape, target.shape)
        all_velocities += batch["velocities"]
        all_diffusivities += batch["diffusivities"]
        
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
    
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
    
        # decode into 100k parameters
        theta = model.decode_theta(theta_latent, dim)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, integration_time=n_output_frames, n_future_steps=n_output_frames, predict_normed=False, metadata=metadata)
    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    print(f"Initial error", rollout_error)

    all_theta_latent.append(theta_latent)
    all_theta.append(theta)
    all_input.append(inp)
    all_target.append(target)


all_theta_latent = torch.stack(all_theta_latent)
all_theta = torch.stack(all_theta)
all_input = torch.stack(all_input)
all_target = torch.stack(all_target)

In [ ]:
dt=1/10
torch.arange(0, 1/10+dt, dt)

In [ ]:
### setup the data 
theta1 = all_theta[0]
theta2 = all_theta[1]
theta3 = all_theta[2]

x_test = all_input[2]
y_test = all_target[2]
state_labels = torch.tensor([0], device=x_test.device)


In [ ]:
#model.max_steps=20

In [ ]:
pred_test = []
x_test_ = x_test[:, -1].clone()
n_output_frames=34
composition_type="composition"

if composition_type=="sum":
    with torch.no_grad():
        for _ in range(n_output_frames):
            pred, _ = model.solve_ode_with_2_operators(x_test_, theta1, theta2, state_labels, dim, integration_time=1, n_future_steps=1, predict_normed=False, metadata=metadata) # Re-initialize metadata for the test run
            pred_test.append(pred[:, -1])
            x_test_ = pred[:, -1]
else:
    with torch.no_grad():
        for _ in range(n_output_frames):
            pred_int, _ = model.solve_ode(x_test_, theta1, state_labels, dim, integration_time=1, n_future_steps=1, predict_normed=False, metadata=metadata)
            pred, _ = model.solve_ode(pred_int[:,-1], theta2, state_labels, dim, integration_time=1, n_future_steps=1, predict_normed=False, metadata=metadata)
            x_test_ = pred[:, -1]
            pred_test.append(pred[:, -1])
pred_test = torch.stack(pred_test, 1)
# Calculate the test error.
test_error = relative_l2_error(pred_test, y_test[:, :n_output_frames], aggregate=None)

In [ ]:
test_error
#idx=0

In [ ]:
for t in range(10):
    plt.plot(y_test.detach().cpu().numpy()[idx, t].squeeze(), label=t)
#plt.legend()

In [ ]:
for t in range(10):
    plt.plot(pred_test.detach().cpu().numpy()[idx, t].squeeze(), label=t)
#plt.legend()

In [ ]:
output_frames=2
index = [1+k*10 for k in range(output_frames+1)]

In [ ]:
index

In [ ]:
x=[t for t in range(21)]
x{index]

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs = 500
n_output_frames=34
composition_type="composition"

theta1_time = []
theta2_time = []

training_horizon=1

# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
#x_test = inp[:, -1].to(device)
#y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
#theta1 = theta1.detach().clone().requires_grad_()
#theta2 = theta2.detach().clone().requires_grad_()

#theta_latent1 = (theta_latent.detach()+0.1*torch.randn_like(theta_latent.detach())).requires_grad_()
#theta_latent2 = (theta_latent.detach()+0.1*torch.randn_like(theta_latent.detach())).requires_grad_()
theta_latent1 = (torch.zeros_like(theta_latent.detach())).requires_grad_()
theta_latent2 = (torch.zeros_like(theta_latent.detach())).requires_grad_()

optimizer = torch.optim.AdamW([theta_latent1] + [theta_latent2], lr=1e-2)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-training_horizon-1)

    theta1 = model.decode_theta(theta_latent1, dim)
    theta2 = model.decode_theta(theta_latent2, dim)

    if composition_type=="sum":
        pred, _ = model.solve_ode_with_2_operators(x_test[:, t], theta1, theta2, state_labels, dim, n_future_steps=training_horizon, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
    else:
        pred=[]
        x_test_ = x_test[:, t].clone()
        for _ in range(training_horizon):
            pred_int, _ = model.solve_ode(x_test_, theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
            pred_, _ = model.solve_ode(pred_int[:, -1], theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
            pred.append(pred_[:, -1])
            x_test_ = pred_[:, -1]
        pred = torch.cat(pred, 1)
    #x_train = inp[:, t]
    #y_train = inp[:, t+1]
    
    # Run the model to get the prediction.
    
    
    # Calculate the training loss.
    loss = relative_l2_error(pred, x_test[:, t+1:t+training_horizon+1])# + 0.001*torch.abs(F.cosine_similarity(theta_latent1, theta_latent2, dim=1)).mean()
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    theta1_time.append(theta_latent1.detach().cpu())
    theta2_time.append(theta_latent2.detach().cpu())
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()
            theta1 = model.decode_theta(theta_latent1, dim)
            theta2 = model.decode_theta(theta_latent2, dim)

            pred_test = []
            x_test_ = x_test[:, -1].clone()
            for _ in range(n_output_frames):
                if composition_type=="sum":
                    pred, _ = model.solve_ode_with_2_operators(x_test_, theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)

                else:
                    pred_int, _ = model.solve_ode(x_test_, theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
                    pred, _ = model.solve_ode(pred_int[:,-1], theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
                pred_test.append(pred)
                x_test_ = pred[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
    
            
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
theta1_time=torch.stack(theta1_time)
theta2_time=torch.stack(theta2_time)

In [ ]:
idx=0

In [ ]:

plt.plot(theta1_time[:, idx, 0], theta1_time[:, idx, 1], linestyle="--",c="gray", zorder=1)
plt.scatter(theta1_time[:, idx, 0], theta1_time[:, idx, 1], c=[t for t in range(theta1_time.shape[0])], zorder=2)
plt.scatter(all_theta_latent[0, idx, 0].cpu().detach(), all_theta_latent[0, idx, 1].cpu().detach(), c="red", label="theta1", zorder=3)
plt.scatter(all_theta_latent[1, idx, 0].cpu().detach(), all_theta_latent[1, idx, 1].cpu().detach(), c="orange", label="theta2", zorder=4)
plt.legend()
plt.colorbar()

In [ ]:

plt.plot(theta2_time[:, idx, 0], theta2_time[:, idx, 1], linestyle="--",c="gray", zorder=1)
plt.scatter(theta2_time[:, idx, 0], theta2_time[:, idx, 1], c=[t for t in range(theta1_time.shape[0])], zorder=2)
plt.scatter(all_theta_latent[0, idx, 0].cpu().detach(), all_theta_latent[0, idx, 1].cpu().detach(), c="red", label="theta1", zorder=3)
plt.scatter(all_theta_latent[1, idx, 0].cpu().detach(), all_theta_latent[1, idx, 1].cpu().detach(), c="orange", label="theta2", zorder=4)
plt.legend()
plt.colorbar()

In [ ]:
model = model.cuda()

In [ ]:
n_batches = 1  # or set as needed for your epoch size
batch_size = 1
split="test"
n_input_frames=16
n_output_frames=34
model.eval()

#composition_type="composition"
#composition_type="sum"

advection_speeds = [0.5, 0., 0.5]
viscosities = [0., 0.5, 0.5]

#advection_speeds = [0.9, 0.9, 1.8]
#advection_speeds = [0.6, 0.1, 0.7]
#viscosities = [0., 0, 0.]

#advection_speeds = [0., 0., 0.]
#viscosities = [0.95, 0.95, 1.9]


all_velocities = []
all_diffusivities = []
test_input = []
test_target = []
test_theta_latent = []

test_ds = TemporalDatasetFixedCI(
        n_batches=n_batches,
        batch_size=batch_size,
        sub_x=sub_x,
        sub_t=sub_t,
        split=split,
        input_frames=n_input_frames,
        output_frames=n_output_frames,
        L=16.0,
        nx=256,
        nt=100,
        T=50.0,
        fractal_power=3.0,
        fractal_degree=256, # nx
        v_range=[0],#(0.01, 1.0),
        D_range=[0], #(0.001, 1.0),
    )


for advection_speed, viscosity in zip(advection_speeds, viscosities):

    test_ds.v_range=[advection_speed]
    test_ds.D_range=[viscosity]
    test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=1, prefetch_factor=1, pin_memory=True, shuffle=False)
    
    for batch in tqdm(test_loader):
        inp, target = batch["input"], batch["target"]
        inp = inp.squeeze(1)
        target = target.squeeze(1)
        print('inp', inp.shape, target.shape)
        all_velocities += batch["velocities"]
        all_diffusivities += batch["diffusivities"]
        
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
    
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
    
        # decode into 100k parameters
        theta = model.decode_theta(theta_latent, dim)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames, predict_normed=False, metadata=metadata)
    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    print(f"Initial error", rollout_error)

    test_theta_latent.append(theta_latent)
    test_input.append(inp)
    test_target.append(target)


test_theta_latent = torch.stack(test_theta_latent)
test_input = torch.stack(test_input)
test_target = torch.stack(test_target)

In [ ]:
test_input.shape

In [ ]:
n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []
all_input = []
all_target = []

for batch in tqdm(train_loader):
    inp, target = batch["input"], batch["target"]
    all_input.append(inp)
    all_target.append(target)
    all_velocities += batch["velocities"]
    all_diffusivities+=batch["diffusivities"]

        
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
    
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    
    # encode into 2 dimensional
    with torch.no_grad():
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
        all_theta.append(theta_latent.cpu().detach())

all_theta = torch.cat(all_theta)
all_velocities = torch.tensor(all_velocities)
all_diffusivities = torch.tensor(all_diffusivities)
all_input = torch.cat(all_input)
all_target = torch.cat(all_target)
#all_velocities = torch.cat(all_velocities)
#all_diffusivities = torch.cat(all_diffusivities)

In [ ]:
theta3 = test_theta_latent[2].cpu().detach()

In [ ]:
plt.scatter(all_theta[:, 0], all_theta[:, 1])
plt.scatter(theta3[0,0], theta3[0,1], c="red")

In [ ]:
def find_k_nearest_neighbors(query_point, data_points, k=1, metric='euclidean'):
    """
    Find k nearest neighbors of a query point in a dataset.
    
    Parameters:
    -----------
    query_point : numpy.ndarray
        Single point of shape (n_features,)
    data_points : numpy.ndarray  
        Dataset of shape (n_samples, n_features)
    k : int, default=1
        Number of nearest neighbors to find
    metric : str, default='euclidean'
        Distance metric ('euclidean', 'manhattan', 'cosine')
    
    Returns:
    --------
    indices : numpy.ndarray
        Indices of k nearest neighbors
    distances : numpy.ndarray
        Distances to k nearest neighbors
    """
    
    if metric == 'euclidean':
        # Euclidean distance: sqrt(sum((a - b)^2))
        distances = np.sqrt(((data_points - query_point)**2).sum(1))
    
    elif metric == 'manhattan':
        # Manhattan distance: sum(|a - b|)
        distances = np.sum(np.abs(data_points - query_point), axis=1)
    
    elif metric == 'cosine':
        # Cosine distance: 1 - cosine_similarity
        query_norm = np.linalg.norm(query_point)
        data_norms = np.linalg.norm(data_points, axis=1)
        dot_products = np.dot(data_points, query_point)
        cosine_similarities = dot_products / (data_norms * query_norm)
        distances = 1 - cosine_similarities
    
    else:
        raise ValueError(f"Unknown metric: {metric}")
    
    # Get indices of k smallest distances
    k_nearest_indices = np.argpartition(distances, k)[:k]
    
    # Sort them by distance (optional, for ordered results)
    k_nearest_indices = k_nearest_indices[np.argsort(distances[k_nearest_indices])]
    
    return k_nearest_indices, distances[k_nearest_indices]

In [ ]:
indices, dist = find_k_nearest_neighbors(theta3, all_theta, k=512)

In [ ]:
plt.scatter(all_theta[indices, 0], all_theta[indices, 1], c= all_diffusivities[indices])
plt.colorbar()
plt.scatter(theta3[0,0], theta3[0,1], c="red")

In [ ]:
plt.scatter(all_theta[indices, 0], all_theta[indices, 1], c= all_velocities[indices])
plt.colorbar()
plt.scatter(theta3[0,0], theta3[0,1], c="red")

In [ ]:
theta = all_theta[indices]
X = all_input[indices]
y = all_target[indices]

theta_test = test_theta_latent[2]
X_test = test_input[2]
y_test = test_target[2]

In [ ]:
X.shape

In [ ]:
batch_indices = torch.randint(0, X.shape[0], size=(batch_size,))
t_indices = torch.randint(0, X.shape[1]-1, size=(batch_size,))
x = X[batch_indices]
torch.gather(x, 1, t_indices.view(-1, 1, 1,1).expand(batch_size,1,X.shape[2], X.shape[3])).shape
#X[batch_indices].shape


In [ ]:
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
device="cuda" if torch.cuda.is_available() else "cpu"
ckpt_time="DISCO_advection-diffusion_adjFalse_h128_t2_steps1_bs64_lr0.0005_ctxTrue_inframes16_outframes2_T50"
ckpt_path = f"/mnt/home/lserrano/disco-ball/outputs/{ckpt_time}/last-v2.ckpt" #last.ckpt" otherwise
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
print(f"Loading model from {ckpt_path}...")
model = DISCOLitModule.load_from_checkpoint(ckpt_path, map_location=device)
model = model.model.to(device)
model.eval()

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs =500
n_output_frames=34
composition_type="composition"
batch_size = 16  # Number of samples to process per iteration

training_horizon=5

# Training data for interpolation.
x_train = X.clone().to(device) #rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
y_train = y.clone().to(device) #urearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)
theta_train = theta.clone().to(device)

# Test data for extrapolation.
x_test = X_test.clone().to(device) #[:, -1].to(device)
y_test = y_test.clone().to(device) #target.to(device)
theta_test = theta_test.clone().to(device)

params_to_optimize = []
params_to_optimize.extend(model.decoder_common.parameters())
params_to_optimize.extend(model.decoder_head.parameters())
params_to_optimize.extend(theta_test)
optimizer = torch.optim.AdamW(params_to_optimize, lr=1e-3)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting hypernetwork finetuning...")
print(f"Optimizing {sum(p.numel() for p in model.hpnn.parameters())} hypernetwork parameters")
print(f"Latent theta are fixed (no gradients)")

# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different behaviors for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    # --- Sample batch indices and time steps ---
    # Sample batch indices
    batch_indices = torch.randint(0, x_train.shape[0], (batch_size,))
    
    # Sample random time steps for X, y (interpolation)
    t_train = torch.randint(0, n_input_frames-training_horizon, (batch_size,)).to(device)
    
    # Sample random time steps for X_test (extrapolation) 
    # These can be different from t_train to add more variety
    t_test = np.random.randint(0, n_input_frames-training_horizon)

    # Decode theta using hypernetwork (this is where gradients flow through hpnn)
    theta_tr = model.decode_theta(theta_train[batch_indices], dim)
    theta_te = model.decode_theta(theta_test, dim)

    x_tr = y_train[batch_indices].clone().to(device)
    x_inp = torch.gather(x_tr, 1, t_train.view(-1, 1, 1,1).expand(batch_size,1,x_tr.shape[2], x_tr.shape[3]))

    # --- Training loss on interpolation data (X, y) with batched sampling ---
    pred_train_list = []
    y_train_list = []

    loss = 0

    x_inp = x_inp[:, 0]
    x_test_ = x_test[:, t_test].clone()
    for j in range(training_horizon):
        x_target = torch.gather(x_tr, 1, j + 1 + t_train.view(-1, 1, 1,1).expand(batch_size,1,x_tr.shape[2], x_tr.shape[3]))

        #print('x_inp', x_inp.shape, x_test_.shape)
        
        pred_train, _ = model.solve_ode(x_inp, theta_tr, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
        pred_test, _ = model.solve_ode(x_test_, theta_te, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)

        # Calculate the combined training loss
        loss_train = relative_l2_error(pred_train, x_target)
        loss_test = relative_l2_error(pred_test, x_test[:, t_test+j+1])

        x_inp = pred_train[:, -1]
        x_test_ = pred_test[:, -1]
        
        # Combine losses (you can adjust the weighting)
        loss += 1.0 *loss_train + 1.0 * loss_test

    loss /= training_horizon

     # --- Backpropagation ---
    # Standard training step - gradients only flow through hypernetwork
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()
            theta_te = model.decode_theta(theta_test, dim)
            pred_test = []
            x_test_ = x_test[:, -1].clone() # x_test[:, -1]
            #for _ in range(n_input_frames-1):
            for _ in range(n_output_frames):
                x_test_, _ = model.solve_ode(x_test_, theta_te, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
                pred_test.append(x_test_)
                x_test_ = x_test_[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
            #test_error = relative_l2_error(pred_test, x_test[:, 1:]).item()
    
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation Loss: {loss_train.item():.6f} | "
            f"Test Loss: {loss_test.item():.6f} | "
            f"Combined Loss: {loss.item():.6f} | "
            f"Extrapolation Error: {test_error:.6f}"
        )

print("Hypernetwork finetuning finished.")

In [ ]:
fig, ax = plot_prediction_comparison(pred_test.squeeze(1), y_test.squeeze(1), idx=0)
#fig, ax = plot_prediction_comparison(pred_test.squeeze(1), x_test[:,1:].squeeze(1), idx=0)

In [ ]:
x_target.shape

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs =500
n_output_frames=34
composition_type="composition"
batch_size = 16  # Number of samples to process per iteration

training_horizon=5

# Training data for interpolation.
x_train = X.clone().to(device) #rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
y_train = y.clone().to(device) #urearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)
theta_train = theta.clone().to(device)

# Test data for extrapolation.
x_test = X_test.clone().to(device) #[:, -1].to(device)
y_test = y_test.clone().to(device) #target.to(device)
theta_test = theta_test.clone().to(device)

params_to_optimize = []
params_to_optimize.extend(model.decoder_common.parameters())
params_to_optimize.extend(model.decoder_head.parameters())
params_to_optimize.extend(theta_test)
optimizer = torch.optim.AdamW(params_to_optimize, lr=1e-3)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting hypernetwork finetuning...")
print(f"Optimizing {sum(p.numel() for p in model.hpnn.parameters())} hypernetwork parameters")
print(f"Latent theta are fixed (no gradients)")

# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different behaviors for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    # --- Sample batch indices and time steps ---
    # Sample batch indices
    batch_indices = torch.randint(0, x_train.shape[0], (batch_size,))
    
    # Sample random time steps for X, y (interpolation)
    t_train = torch.randint(0, n_input_frames-training_horizon, (batch_size,)).to(device)
    
    # Sample random time steps for X_test (extrapolation) 
    # These can be different from t_train to add more variety
    t_test = np.random.randint(0, n_input_frames-training_horizon)

    # Decode theta using hypernetwork (this is where gradients flow through hpnn)
    theta_tr = model.decode_theta(theta_train[batch_indices], dim)
    theta_te = model.decode_theta(theta_test, dim)

    x_tr = y_train[batch_indices].clone().to(device)
    x_inp = torch.gather(x_tr, 1, t_train.view(-1, 1, 1,1).expand(batch_size,1,x_tr.shape[2], x_tr.shape[3]))

    # --- Training loss on interpolation data (X, y) with batched sampling ---
    pred_train_list = []
    y_train_list = []

    loss = 0

    x_inp = x_inp[:, 0]
    x_test_ = x_test[:, t_test].clone()
    for j in range(training_horizon):
        x_target = torch.gather(x_tr, 1, j + 1 + t_train.view(-1, 1, 1,1).expand(batch_size,1,x_tr.shape[2], x_tr.shape[3]))

        #print('x_inp', x_inp.shape, x_test_.shape)
        
        pred_train, _ = model.solve_ode(x_inp, theta_tr, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
        pred_test, _ = model.solve_ode(x_test_, theta_te, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)

        # Calculate the combined training loss
        loss_train = relative_l2_error(pred_train, x_target)
        loss_test = relative_l2_error(pred_test, x_test[:, t_test+j+1])

        x_inp = pred_train[:, -1]
        x_test_ = pred_test[:, -1]
        
        # Combine losses (you can adjust the weighting)
        loss += 1.0 *loss_train + 1.0 * loss_test

    loss /= training_horizon

     # --- Backpropagation ---
    # Standard training step - gradients only flow through hypernetwork
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()
            theta_te = model.decode_theta(theta_test, dim)
            pred_test = []
            x_test_ = x_test[:, -1].clone() # x_test[:, -1]
            #for _ in range(n_input_frames-1):
            for _ in range(n_output_frames):
                x_test_, _ = model.solve_ode(x_test_, theta_te, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
                pred_test.append(x_test_)
                x_test_ = x_test_[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
            #test_error = relative_l2_error(pred_test, x_test[:, 1:]).item()
    
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation Loss: {loss_train.item():.6f} | "
            f"Test Loss: {loss_test.item():.6f} | "
            f"Combined Loss: {loss.item():.6f} | "
            f"Extrapolation Error: {test_error:.6f}"
        )

print("Hypernetwork finetuning finished.")